In [14]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor

In [15]:
# Here we load in the tables that we are going to be using for the rest of the downstream code 
seed_predictor = TabularPredictor.load('C:/Users/veeri/Documents\Coding_Projects\MM_2026\Code_Production\mm_autogluon_model_historical')
predictor = TabularPredictor.load('C:/Users/veeri/Documents\Coding_Projects\MM_2026\Code_Production\mm_autogluon_model_2026_inseason')
tourney = pd.read_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Code_Production/tourney_processed.csv')
team_games = pd.read_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Code_Production/team_games_processed.csv')

# Here are the weightings that I chose for the final model
# My rational is that the performance of the team coming into MM has a large impact on the overall output of the games
# However, the selection committee usually weights the schedule and quadrant wins into the seeding of a team 
# I also wanted to get some historical information about the team within this model 
UPSET_THRESHOLD = 0.58   # derived from the in-season model's calibration
IN_SEASON_WEIGHT = 0.63  # the smarter model gets more weight
SEED_WEIGHT = 0.37     # stabilizer that leans toward favorites

# I wanted to map the bracket names to what is actually within our dataset 
# i.e. VCU is the actual name on the bracket, but in the dataset it is va commonwealth 
bracket_to_canonical = {
    # East Region 
    'Duke':'duke',
    'Siena':'siena',
    'Ohio State':'ohio st',
    'TCU':'tcu',
    "St John's":"saint john`s",
    'Northern Iowa':'northern iowa',
    'Kansas':'kansas',
    'CA Baptist':'cal baptist',
    'Louisville':'louisville',
    'South Florida':'south fla.',
    'Michigan St':'michigan st',
    'N Dakota St':'n dakota st',
    'UCLA':'ucla',
    'UCF':'central florida',
    'UConn':'connecticut',
    'Furman':'furman',

    # South Region 
    'Florida':'florida',
    'Prairie View A&M':'prairie view',
    'Clemson':'clemson',
    'Iowa':'iowa',
    'Vanderbilt':'vanderbilt',
    'McNeese':'mcneese',
    'Nebraska':'nebraska',
    'Troy':'troy',
    'North Carolina':'north carolina',
    'VCU':'va commonwealth',
    'Illinois':'illinois',
    'Penn':'penn',
    "Saint Mary's":"saint mary's",
    'Texas A&M':'texas a&m',
    'Houston':'houston',
    'Idaho':'idaho',

    # West Region 
    'Arizona':'arizona',
    'Long Island':None,  # not in our data — seed model only
    'Villanova':'villanova',
    'Utah St':'utah st',
    'Wisconsin':'wisconsin',
    'High Point':'high point',
    'Arkansas':'arkansas',
    'Hawaii':"hawai'i",
    'BYU':'brigham young',
    'Texas':'texas',  # play-in winner, beat NC State
    'Gonzaga':'gonzaga',
    'Kennesaw St':'kennesaw',
    'Miami (FL)':'miami (fl)',
    'Missouri':'missouri',
    'Purdue':'purdue',
    'Queens (NC)':'queens (nc)',

    # Midwest Region
    'Michigan':'michigan',
    'Howard':'howard',  # play-in winner, beat UMBC
    'Georgia':'georgia',
    'Saint Louis':'saint louis',
    'Texas Tech':'texas tech',
    'Akron':'akron',
    'Alabama':'alabama',
    'Hofstra':'hofstra',
    'Tennessee':'tennessee',
    'Miami (OH)':'miami (oh)',  # play-in winner, beat SMU
    'Virginia':'virginia',
    'Wright St':'wright st',
    'Kentucky':'kentucky',
    'Santa Clara':'santa clara',
    'Iowa St':'iowa st',
    'Tennessee St':'tennessee st',
}




<>:2: SyntaxWarning: invalid escape sequence '\C'
<>:3: SyntaxWarning: invalid escape sequence '\C'
<>:2: SyntaxWarning: invalid escape sequence '\C'
<>:3: SyntaxWarning: invalid escape sequence '\C'
C:\Users\veeri\AppData\Local\Temp\ipykernel_33268\779751066.py:2: SyntaxWarning: invalid escape sequence '\C'
  seed_predictor = TabularPredictor.load('C:/Users/veeri/Documents\Coding_Projects\MM_2026\Code_Production\mm_autogluon_model_historical')
C:\Users\veeri\AppData\Local\Temp\ipykernel_33268\779751066.py:3: SyntaxWarning: invalid escape sequence '\C'
  predictor = TabularPredictor.load('C:/Users/veeri/Documents\Coding_Projects\MM_2026\Code_Production\mm_autogluon_model_2026_inseason')


In [16]:
# Check the play-in winners in our data
for keyword in ['miami', 'ohio']:
    matches = team_games[team_games['TeamName'].str.contains(keyword,case=False,na=False)]['TeamName'].unique()
    print(f"'{keyword}': {matches.tolist()}")

'miami': ['miami (fl)', 'miami (oh)']
'ohio': ['ohio', 'ohio st']


In [17]:

def get_team_stats(team_name):
    """
    Purpose: Using the bracket_to_canonical dictionary, given a team name, we can get the 
    team's season stats 

    Input: team_name - The team name that we find on the bracket 
    
    """
    # Here we translate the bracket name to the name in our data. For example 
    # 'BYU' is used to get 'brigham young'. We use the names on ESPN, but the data 
    # has different values 
    canonical = bracket_to_canonical.get(team_name)
    # If the team is not in the mapping, then we will set it to NONE 
    if canonical is None:
        return None
    # Here we find every row in team_games for this team. team_games has one row per team per 
    # game sorted chronologically. For example, Kansas may have 30 rows, one for each game that they have plaed 
    # Each row has the rolling averages computed from all the prior games 
    team_rows = team_games[team_games['TeamName'] == canonical]
    if len(team_rows) == 0:
        return None
    # The key line! Here we grab the last row, which is their most recent game. The rolling stats 
    # reflect everything that came before it. The last row contains the most complete season profile. 
    return team_rows.iloc[-1]

def predict_matchup(fav_team, fav_seed, und_team, und_seed, round_num):
    """
    Predict a single matchup using the ensemble of seed + in-season models.
    Input
        fav_team: The name of the team that is the favorite 
        fav_seed: The seed of the favorite team 
        und_team: The name of the team that is the underdog 
        und_seedL The seed of the team that is the underdog 
    Returns: (winner_name, winner_seed, prob_fav_win)
    """

    ####################### Historical Model #############################
    # This looks through all historical tournament data and finds every time this exact 
    # seed matchup has happened before. For example, for a 1 v 16 seed, it finds every historical 
    # 1v 16 matchup and calculates the percentage the 1 seed won. 
    # We also get the times this matchup has happened 
    matchup = tourney[(tourney['FavSeed'] == fav_seed) & (tourney['UndSeed'] == und_seed)]
    hist_winrate = matchup['FavWin'].mean() if len(matchup) > 0 else 0.65
    hist_count = len(matchup)

    # Here we answer the question of how does this faavorite's mseed do against everyone, not 
    # just this specific underdog seed? For example, a 1 seed's overall win rate across all matchups
    fav_games = tourney[tourney['FavSeed'] == fav_seed]
    fav_rate = fav_games['FavWin'].mean() if len(fav_games) > 0 else 0.65

    # Now the flip, how often does this underdog seed pull an upset against anyone? 
    und_games = tourney[tourney['UndSeed'] == und_seed]
    und_upset = (1 - und_games['FavWin'].mean()) if len(und_games) > 0 else 0.35

    # Here we create a nice, tidy dataframe with all the features necessary to get the model 
    # prediction. We capture all features that the model was trained on. 
    seed_input = pd.DataFrame([{
        'FavSeed': fav_seed, 'UndSeed': und_seed,
        'SeedDiff': und_seed - fav_seed, 'Round': round_num,
        'HistWinRate': hist_winrate, 'HistMatchupCount': hist_count,
        'FavSeedHistWinRate': fav_rate, 'UndSeedHistUpsetRate': und_upset,
    }])

    # Here we use the best model to run inference
    # We use [1] to get the prbability of the favorite winning 
    seed_prob = seed_predictor.predict_proba(seed_input)[1].values[0]


    ####################### In-Season Model #############################

    # Here we look up each team's most recent rolling stats from team_games. This returns the team's
    # season average for point differential, win rate, blocks, rebounds, etc. 
    fav_stats = get_team_stats(fav_team)
    und_stats = get_team_stats(und_team)

    # Only proceed if we have data for both teams 
    if fav_stats is not None and und_stats is not None:
        # Here we build the matchup features the same way that we did during the training process. 
        # All Diff_ fields are the favorite stat - the underdog stat 
        # A_ is the favorite's stats 
        # B_ is the underdog's stats 
        in_season_input = pd.DataFrame([{
            'Diff_Roll_PointDiff':fav_stats['Roll_PointDiff']-und_stats['Roll_PointDiff'],
            'A_Q1Q2_Losses':fav_stats['Q1Q2_Losses'],
            'A_Roll_OppScore':fav_stats['Roll_OppScore'],
            'Diff_Roll_Stl':fav_stats['Roll_Stl']-und_stats['Roll_Stl'],
            'Diff_Roll_OR':fav_stats['Roll_OR']-und_stats['Roll_OR'],
            'A_Roll_PointDiff':fav_stats['Roll_PointDiff'],
            'B_L10_FGPct':und_stats['L10_FGPct'],
            'A_L10_Stl':fav_stats['L10_Stl'],
            'Diff_L10_OppScore':fav_stats['L10_OppScore']-und_stats['L10_OppScore'],
            'Diff_Roll_OppFGPct':fav_stats['Roll_OppFGPct']-und_stats['Roll_OppFGPct'],
            'A_Roll_Stl':fav_stats['Roll_Stl'],
            'A_L10_TotalReb':fav_stats['L10_TotalReb'],
            'A_Roll_OR':fav_stats['Roll_OR'],
            'A_L10_Blk':fav_stats['L10_Blk'],
            'A_Roll_FGM':fav_stats['Roll_FGM'],
            'A_Q1_Wins':fav_stats['Q1_Wins'],
            'Diff_L10_TO':fav_stats['L10_TO']-und_stats['L10_TO'],
            'A_Roll_OppFGPct':fav_stats['Roll_OppFGPct'],
            'B_L10_Blk':und_stats['L10_Blk'],
            'A_L10_OR':fav_stats['L10_OR'],
            'B_Roll_PointDiff':und_stats['Roll_PointDiff'],
            'B_L10_FGA3':und_stats['L10_FGA3'],
            'A_Roll_Blk':fav_stats['Roll_Blk'],
            'A_L10_FGA3':fav_stats['L10_FGA3'],
            'B_L10_FGA':und_stats['L10_FGA'],
            'B_L10_FTPct':und_stats['L10_FTPct'],
            'A_Q2_Losses':fav_stats['Q2_Losses'],
            'Diff_Q3Q4_Losses':fav_stats['Q3Q4_Losses']-und_stats['Q3Q4_Losses'],
            'B_L10_FGM':und_stats['L10_FGM'],
            'Diff_L10_OR':fav_stats['L10_OR']-und_stats['L10_OR'],
            'B_Roll_Blk':und_stats['Roll_Blk'],
        }])
        # Use the in-season model 
        in_season_prob = predictor.predict_proba(in_season_input)[1].values[0]

        # The final probability will be the blend of 60% in-season model and 40% historical 
        # model 
        final_prob = IN_SEASON_WEIGHT * in_season_prob + SEED_WEIGHT * seed_prob
    else:
        final_prob = seed_prob

    # If the final_prob is greater than the upset threshold, then we return the 
    # favorite, otherwise the underdog 
    if final_prob >= UPSET_THRESHOLD:
        return fav_team, fav_seed, final_prob
    else:
        return und_team, und_seed, final_prob


In [18]:
regions = {
    'East': [
        {'FavSeed': 1,  'FavTeam': 'Duke',         'UndSeed': 16, 'UndTeam': 'Siena'},
        {'FavSeed': 8,  'FavTeam': 'Ohio State',   'UndSeed': 9,  'UndTeam': 'TCU'},
        {'FavSeed': 5,  'FavTeam': "St John's",    'UndSeed': 12, 'UndTeam': 'Northern Iowa'},
        {'FavSeed': 4,  'FavTeam': 'Kansas',        'UndSeed': 13, 'UndTeam': 'CA Baptist'},
        {'FavSeed': 6,  'FavTeam': 'Louisville',    'UndSeed': 11, 'UndTeam': 'South Florida'},
        {'FavSeed': 3,  'FavTeam': 'Michigan St',   'UndSeed': 14, 'UndTeam': 'N Dakota St'},
        {'FavSeed': 7,  'FavTeam': 'UCLA',          'UndSeed': 10, 'UndTeam': 'UCF'},
        {'FavSeed': 2,  'FavTeam': 'UConn',         'UndSeed': 15, 'UndTeam': 'Furman'},
    ],
    'South': [
        {'FavSeed': 1,  'FavTeam': 'Florida',       'UndSeed': 16, 'UndTeam': 'Prairie View A&M'},
        {'FavSeed': 8,  'FavTeam': 'Clemson',       'UndSeed': 9,  'UndTeam': 'Iowa'},
        {'FavSeed': 5,  'FavTeam': 'Vanderbilt',    'UndSeed': 12, 'UndTeam': 'McNeese'},
        {'FavSeed': 4,  'FavTeam': 'Nebraska',      'UndSeed': 13, 'UndTeam': 'Troy'},
        {'FavSeed': 6,  'FavTeam': 'North Carolina', 'UndSeed': 11, 'UndTeam': 'VCU'},
        {'FavSeed': 3,  'FavTeam': 'Illinois',      'UndSeed': 14, 'UndTeam': 'Penn'},
        {'FavSeed': 7,  'FavTeam': "Saint Mary's",  'UndSeed': 10, 'UndTeam': 'Texas A&M'},
        {'FavSeed': 2,  'FavTeam': 'Houston',       'UndSeed': 15, 'UndTeam': 'Idaho'},
    ],
    'West': [
        {'FavSeed': 1,  'FavTeam': 'Arizona',       'UndSeed': 16, 'UndTeam': 'Long Island'},
        {'FavSeed': 8,  'FavTeam': 'Villanova',     'UndSeed': 9,  'UndTeam': 'Utah St'},
        {'FavSeed': 5,  'FavTeam': 'Wisconsin',     'UndSeed': 12, 'UndTeam': 'High Point'},
        {'FavSeed': 4,  'FavTeam': 'Arkansas',      'UndSeed': 13, 'UndTeam': 'Hawaii'},
        {'FavSeed': 6,  'FavTeam': 'BYU',           'UndSeed': 11, 'UndTeam': 'Texas'},
        {'FavSeed': 3,  'FavTeam': 'Gonzaga',       'UndSeed': 14, 'UndTeam': 'Kennesaw St'},
        {'FavSeed': 7,  'FavTeam': 'Miami (FL)',    'UndSeed': 10, 'UndTeam': 'Missouri'},
        {'FavSeed': 2,  'FavTeam': 'Purdue',        'UndSeed': 15, 'UndTeam': 'Queens (NC)'},
    ],
    'Midwest': [
        {'FavSeed': 1,  'FavTeam': 'Michigan',      'UndSeed': 16, 'UndTeam': 'Howard'},
        {'FavSeed': 8,  'FavTeam': 'Georgia',       'UndSeed': 9,  'UndTeam': 'Saint Louis'},
        {'FavSeed': 5,  'FavTeam': 'Texas Tech',    'UndSeed': 12, 'UndTeam': 'Akron'},
        {'FavSeed': 4,  'FavTeam': 'Alabama',       'UndSeed': 13, 'UndTeam': 'Hofstra'},
        {'FavSeed': 6,  'FavTeam': 'Tennessee',     'UndSeed': 11, 'UndTeam': 'Miami (OH)'},  # play-in winner
        {'FavSeed': 3,  'FavTeam': 'Virginia',      'UndSeed': 14, 'UndTeam': 'Wright St'},
        {'FavSeed': 7,  'FavTeam': 'Kentucky',      'UndSeed': 10, 'UndTeam': 'Santa Clara'},
        {'FavSeed': 2,  'FavTeam': 'Iowa St',       'UndSeed': 15, 'UndTeam': 'Tennessee St'},
    ],
}

round_names = {1: 'ROUND OF 64', 2: 'ROUND OF 32', 3: 'SWEET 16',
               4: 'ELITE 8', 5: 'FINAL FOUR', 6: 'CHAMPIONSHIP'}

In [19]:
# Here we want to simulate the full tournament, round by round
# We store each region's champion so they can play in the Final Four
regional_champs = {}

# Loop through each of the 4 regions (East, South, West, Midwest)
for region_name, games in regions.items():
    # This list holds the winners from the current round
    # After Round 1 it has 8 teams, after Round 2 it has 4, etc.
    current_round_teams = []

    ############################ Round 1 ###################
    print(f"\n{'═' * 70}")
    print(f"  {region_name.upper()} REGION — {round_names[1]}")
    print(f"{'═' * 70}")

    # Play each of the 8 first-round games in the region
    for game in games:
        # Use our ensemble model to predict who wins
        winner, winner_seed, prob = predict_matchup(
            game['FavTeam'], game['FavSeed'], game['UndTeam'], game['UndSeed'], 1
        )
        # Check if our model picked the underdog
        is_upset = winner == game['UndTeam']
        flag = '🔥 UPSET' if is_upset else ''
        label = ('UPSET' if is_upset else
                 'LOCK' if prob >= 0.8 else
                 'STRONG' if prob >= 0.7 else 'LEAN')

        print(f"  ({game['FavSeed']:>2}) {game['FavTeam']:<20} vs ({game['UndSeed']:>2}) {game['UndTeam']:<20} "
              f"→ {winner:<20} [{label}] (Fav: {prob:.1%}) {flag}")

        # Store the winner so they can play in the next round
        current_round_teams.append({'Team': winner, 'Seed': winner_seed})

    ##################### Rounds 2-4 (within region) ###############################
    # Round 2 = Round of 32, Round 3 = Sweet 16, Round 4 = Elite 8
    # Each round takes the winners from the previous round and pairs them up
    for round_num in range(2, 5):
        print(f"\n{'─' * 50}")
        print(f"  {region_name.upper()} — {round_names[round_num]}")
        print(f"{'─' * 50}")

        next_round_teams = []
        # Pair up winners: game 0 vs game 1, game 2 vs game 3, etc.
        # This follows the standard NCAA bracket structure where
        # the 1-seed side of the bracket eventually meets the 2-seed side
        for i in range(0, len(current_round_teams), 2):
            team_a = current_round_teams[i]
            team_b = current_round_teams[i + 1]

            # Determine who is the favorite based on seed
            # Lower seed number = better team = favorite
            if team_a['Seed'] <= team_b['Seed']:
                fav_team, fav_seed = team_a['Team'], team_a['Seed']
                und_team, und_seed = team_b['Team'], team_b['Seed']
            else:
                fav_team, fav_seed = team_b['Team'], team_b['Seed']
                und_team, und_seed = team_a['Team'], team_a['Seed']

            # Predict this matchup using our ensemble
            winner, winner_seed, prob = predict_matchup(
                fav_team, fav_seed, und_team, und_seed, round_num
            )
            is_upset = winner == und_team
            flag = '🔥 UPSET' if is_upset else ''

            print(f"  ({fav_seed:>2}) {fav_team:<20} vs ({und_seed:>2}) {und_team:<20} "
                  f"→ {winner:<20} (Fav: {prob:.1%}) {flag}")

            next_round_teams.append({'Team': winner, 'Seed': winner_seed})

        # The winners of this round become the input for the next round
        current_round_teams = next_round_teams

    # After Round 4 (Elite 8), only 1 team remains per region
    # That team is the regional champion and advances to the Final Four
    regional_champs[region_name] = current_round_teams[0]

# ################## Final Four ########################
# The standard NCAA bracket pairs: East vs West, South vs Midwest
print(f"\n{'═' * 70}")
print(f"  {round_names[5]}")
print(f"{'═' * 70}")

ff_matchups = [('East', 'West'), ('South', 'Midwest')]
championship_teams = []

for region_a, region_b in ff_matchups:
    team_a = regional_champs[region_a]
    team_b = regional_champs[region_b]

    # Same logic: lower seed = favorite
    if team_a['Seed'] <= team_b['Seed']:
        fav_team, fav_seed = team_a['Team'], team_a['Seed']
        und_team, und_seed = team_b['Team'], team_b['Seed']
    else:
        fav_team, fav_seed = team_b['Team'], team_b['Seed']
        und_team, und_seed = team_a['Team'], team_a['Seed']

    winner, winner_seed, prob = predict_matchup(
        fav_team, fav_seed, und_team, und_seed, 5
    )
    is_upset = winner == und_team
    flag = '🔥 UPSET' if is_upset else ''

    print(f"  {region_a} vs {region_b}:")
    print(f"  ({fav_seed:>2}) {fav_team:<20} vs ({und_seed:>2}) {und_team:<20} "
          f"→ {winner:<20} (Fav: {prob:.1%}) {flag}")

    # Store the Final Four winners for the championship game
    championship_teams.append({'Team': winner, 'Seed': winner_seed})

################# NATYYYYY ############################
# The two Final Four winners play for the national title
print(f"\n{'═' * 70}")
print(f"  {round_names[6]}")
print(f"{'═' * 70}")

team_a = championship_teams[0]
team_b = championship_teams[1]

if team_a['Seed'] <= team_b['Seed']:
    fav_team, fav_seed = team_a['Team'], team_a['Seed']
    und_team, und_seed = team_b['Team'], team_b['Seed']
else:
    fav_team, fav_seed = team_b['Team'], team_b['Seed']
    und_team, und_seed = team_a['Team'], team_a['Seed']

winner, winner_seed, prob = predict_matchup(
    fav_team, fav_seed, und_team, und_seed, 6
)
is_upset = winner == und_team
flag = '🔥 UPSET' if is_upset else ''

print(f"  ({fav_seed:>2}) {fav_team:<20} vs ({und_seed:>2}) {und_team:<20} "
      f"→ {winner:<20} (Fav: {prob:.1%}) {flag}")

print(f"\n{'═' * 70}")
print(f"  🏆 NATIONAL CHAMPION: ({winner_seed}) {winner} 🏆")
print(f"{'═' * 70}")


══════════════════════════════════════════════════════════════════════
  EAST REGION — ROUND OF 64
══════════════════════════════════════════════════════════════════════
  ( 1) Duke                 vs (16) Siena                → Duke                 [LOCK] (Fav: 91.4%) 
  ( 8) Ohio State           vs ( 9) TCU                  → TCU                  [UPSET] (Fav: 49.5%) 🔥 UPSET
  ( 5) St John's            vs (12) Northern Iowa        → St John's            [LEAN] (Fav: 67.3%) 
  ( 4) Kansas               vs (13) CA Baptist           → Kansas               [LEAN] (Fav: 60.5%) 
  ( 6) Louisville           vs (11) South Florida        → South Florida        [UPSET] (Fav: 51.1%) 🔥 UPSET
  ( 3) Michigan St          vs (14) N Dakota St          → Michigan St          [LEAN] (Fav: 69.9%) 
  ( 7) UCLA                 vs (10) UCF                  → UCLA                 [LEAN] (Fav: 59.2%) 
  ( 2) UConn                vs (15) Furman               → UConn                [LOCK] (Fav: 84.3%) 

────